In [ ]:
# 필요한 라이브러리 설치
!pip install torch torchvision torchaudio transformers langchain langchain-community langchain-huggingface langchain-qdrant qdrant-client sentence-transformers rank_bm25 pydantic openai scikit-learn

  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_nvtx_cu12-12.1.105-py3-none-manylinu

In [ ]:
import torch
import torchvision
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client import models
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_qdrant import Qdrant as LangchainQdrant
from langchain.llms import HuggingFacePipeline
from langchain.embeddings import OpenAIEmbeddings
from rank_bm25 import BM25Okapi
from typing import List, Tuple
import logging
import numpy as np
import time
from sklearn.metrics.pairwise import cosine_similarity
import os
from google.colab import userdata

In [ ]:
# 로깅 설정
logging.basicConfig(level=logging.DEBUG, format='%(asctime)s - %(levelname)s - %(message)s')

# Qdrant 클라이언트 초기화
QDRANT_URL = "https://6e46b2c2-f28a-4f28-854d-432ab699fdfd.europe-west3-0.gcp.cloud.qdrant.io"
QDRANT_API_KEY = "u2eejPgTwIyhr7BVjFBtkjGdGYPWvzQTBkoYycErtm5cyrFjwEEH9w"
COLLECTION_NAME = "son5"

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

# 임베딩 모델 초기화
embeddings = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")

# Langchain Qdrant 초기화
vector_store = LangchainQdrant(
    client=client,
    collection_name=COLLECTION_NAME,
    embeddings=embeddings
)

/usr/local/lib/python3.10/dist-packages/langchain_core/_api/deprecation.py:141: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  warn_deprecated(
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.86k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/langchain_core/_api/deprecation.py:141: LangChainDeprecationWarning: The class `Qdrant` was deprecated in LangChain 0.1.2 and will be removed in 0.5.0. Use QdrantVectorStore instead.
  warn_deprecated(


In [ ]:
# 샘플 모델 초기화
model_name = "centwon/ko-gpt-trinity-cw"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to('cuda' if torch.cuda.is_available() else 'cpu')

# OpenAI API 키 설정
os.environ["OPENAI_API_KEY"] = userdata.get('FINAL_TEAM3')

# 파이프라인 생성
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_length=256,
    do_sample=True,
    temperature=0.7,
    top_p=0.92,
    no_repeat_ngram_size=2,
    device=0 if torch.cuda.is_available() else -1,
    truncation=True
)

tokenizer_config.json:   0%|          | 0.00/64.0k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/732k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/233k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.64M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/577 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/868 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [ ]:
# LLM 초기화
llm = HuggingFacePipeline(pipeline=pipe)

# 프롬프트 템플릿 정의
prompt_template = """
질문: {question}
관련 카테고리: {category}
관련 질병: {disease}
사용자 의도: {intent}
컨텍스트: {context}

위 정보를 바탕으로 답변해주세요:
"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["question", "category", "disease", "intent", "context"]
)

/usr/local/lib/python3.10/dist-packages/langchain_core/_api/deprecation.py:141: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 0.3. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFacePipeline`.
  warn_deprecated(


In [ ]:
# RetrievalQA 체인 생성
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vector_store.as_retriever(search_kwargs={"k": 5}),
    return_source_documents=True,
    chain_type_kwargs={"prompt": PROMPT}
)

# 유사도 임계값 설정
SIMILARITY_THRESHOLD = 0.5

In [ ]:
# Sentence Transformer 모델
st_model = SentenceTransformer('jhgan/ko-sroberta-multitask')

# OpenAI Embedding 모델
openai_embeddings = OpenAIEmbeddings()

/usr/local/lib/python3.10/dist-packages/langchain_core/_api/deprecation.py:141: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 0.3.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import OpenAIEmbeddings`.
  warn_deprecated(


In [ ]:
def get_unique_metadata():
    try:
        response = client.scroll(
            collection_name=COLLECTION_NAME,
            scroll_filter=None,
            limit=10000,
            with_payload=True,
            with_vectors=False
        )

        categories = set()
        diseases = set()
        intents = set()

        for point in response[0]:
            category = point.payload.get("질병_카테고리")
            disease = point.payload.get("질병")
            intent = point.payload.get("의도")

            if category:
                categories.add(category)
            if disease:
                diseases.add(disease)
            if intent:
                intents.add(intent)

        return list(categories), list(diseases), list(intents)
    except Exception as e:
        logging.error(f"메타데이터 가져오기 중 오류 발생: {str(e)}")
        return [], [], []

DISEASE_CATEGORIES, DISEASES, INTENTS = get_unique_metadata()


In [ ]:
def extract_metadata(query, categories, diseases, intents):
    query_lower = query.lower()
    category = next((cat for cat in categories if cat.lower() in query_lower), None)
    disease = next((dis for dis in diseases if dis.lower() in query_lower), None)
    intent = next((intent for intent in intents if intent.lower() in query_lower), None)
    logging.debug(f"추출된 메타데이터: 카테고리={category}, 질병={disease}, 의도={intent}")
    return category, disease, intent


In [ ]:
def expand_query(query: str) -> List[str]:
    expanded_queries = [query]

    synonyms = {
        "증상": ["징후", "병증", "증세"],
        "치료": ["처치", "요법", "치료법"],
        "예방": ["방지", "예방책", "예방법"],
        "통증": ["아픔", "고통", "불편함"],
    }

    words = query.split()
    for word in words:
        if word in synonyms:
            for syn in synonyms[word]:
                expanded_queries.append(query.replace(word, syn))

    category, disease, intent = extract_metadata(query, DISEASE_CATEGORIES, DISEASES, INTENTS)
    if category:
        expanded_queries.append(f"{query} {category}")
    if disease:
        expanded_queries.append(f"{query} {disease}")
    if intent:
        expanded_queries.append(f"{query} {intent}")

    logging.debug(f"확장된 쿼리: {expanded_queries}")
    return list(set(expanded_queries))

In [ ]:
# tiktoken 설치
!pip install tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 5.6 MB/s eta 0:00:00


In [ ]:
import tiktoken
from langchain.embeddings import OpenAIEmbeddings
from typing import List, Dict, Any

In [ ]:
def bm25_search(query: str, documents: List[Any], top_k: int = 5) -> List[Any]:
    if not documents:
        logging.warning("BM25 검색: 문서 목록이 비어 있습니다.")
        return []
    try:
        corpus = [doc.page_content for doc in documents if hasattr(doc, 'page_content') and doc.page_content.strip()]
        if not corpus:
            logging.warning("BM25 검색: 모든 문서의 내용이 비어 있습니다.")
            return []
        bm25 = BM25Okapi(corpus)
        scores = bm25.get_scores(query.split())
        top_results = sorted(zip(documents, scores), key=lambda x: x[1], reverse=True)[:top_k]
        return [doc for doc, score in top_results]
    except Exception as e:
        logging.error(f"BM25 검색 중 오류 발생: {str(e)}")
        return []

In [ ]:
def vector_search(query, documents, top_k=5):
    if not documents:
        logging.warning("벡터 검색: 문서 목록이 비어 있습니다.")
        return []
    query_embedding = st_model.encode([query])[0]
    doc_embeddings = st_model.encode([doc.page_content for doc in documents])
    similarities = cosine_similarity([query_embedding], doc_embeddings)[0]
    top_results = sorted(zip(documents, similarities), key=lambda x: x[1], reverse=True)[:top_k]
    return [doc for doc, score in top_results]


In [ ]:
def openai_search(query, documents, top_k=5):
    if not documents:
        logging.warning("OpenAI 검색: 문서 목록이 비어 있습니다.")
        return []
    query_embedding = openai_embeddings.embed_query(query)
    doc_embeddings = openai_embeddings.embed_documents([doc.page_content for doc in documents])
    similarities = cosine_similarity([query_embedding], doc_embeddings)[0]
    top_results = sorted(zip(documents, similarities), key=lambda x: x[1], reverse=True)[:top_k]
    return [doc for doc, score in top_results]


In [ ]:
def ensemble_search(query: str, documents: List[Any], top_k: int = 5) -> List[Any]:
    if not documents:
        logging.warning("앙상블 검색: 문서 목록이 비어 있습니다.")
        return []
    bm25_results = bm25_search(query, documents, top_k)
    vector_results = vector_search(query, documents, top_k)
    openai_results = openai_search(query, documents, top_k)

    # 결과를 리스트로 유지하고 중복을 제거합니다.
    combined_results = list(dict.fromkeys(bm25_results + vector_results + openai_results))

    if not combined_results:
        logging.warning("앙상블 검색: 결과가 없습니다.")
        return []

    # 최종 순위 결정을 위해 BM25 점수 사용
    corpus = [doc.page_content for doc in combined_results if hasattr(doc, 'page_content')]
    if not corpus:
        logging.warning("앙상블 검색: 모든 문서의 내용이 비어 있습니다.")
        return []
    bm25 = BM25Okapi(corpus)
    scores = bm25.get_scores(query.split())
    sorted_results = sorted(zip(combined_results, scores), key=lambda x: x[1], reverse=True)

    return [doc for doc, _ in sorted_results[:top_k]]


In [ ]:
def generate_llm_response(query: str, context: str = None, use_db: bool = True, metadata: Dict[str, str] = None, fast_mode: bool = True) -> str:
    try:
        max_length = 128 if fast_mode else 256

        if use_db and context:
            category = metadata.get('질병_카테고리', '') if metadata else ''
            disease = metadata.get('질병', '') if metadata else ''
            intent = metadata.get('의도', '') if metadata else ''
        else:
            category = ''
            disease = ''
            intent = ''

        result = qa_chain.invoke({
            "question": query,
            "context": context or "",
            "category": category,
            "disease": disease,
            "intent": intent
        })

        answer = result['result'][:max_length]  # 응답 길이 제한
        print("\nDB 참고 생성된 답변:" if use_db and context else "\nDB를 참고하지 않고 생성된 답변:")
        print(answer)

        return answer
    except Exception as e:
        logging.error(f"LLM 응답 생성 중 오류 발생: {str(e)}")
        return f"죄송합니다. 답변을 생성하는 중에 오류가 발생했습니다: {str(e)}"


In [ ]:
def process_query(query: str, fast_mode: bool = True) -> str:
    start_time = time.time()

    logging.info(f"처리 중인 쿼리: {query}")
    category, disease, intent = extract_metadata(query, DISEASE_CATEGORIES, DISEASES, INTENTS)
    logging.info(f"추출된 메타데이터: 카테고리={category}, 질병={disease}, 의도={intent}")

    expanded_queries = expand_query(query)
    logging.info(f"확장된 쿼리: {expanded_queries}")

    try:
        all_documents = vector_store.similarity_search(query, k=20)

        if not all_documents:
            logging.warning("Qdrant에서 검색 결과가 없습니다.")
            return generate_llm_response(query, use_db=False, fast_mode=fast_mode)

        ensemble_results = ensemble_search(query, all_documents, top_k=5)

        if not ensemble_results:
            logging.warning("앙상블 검색 결과가 없습니다.")
            return generate_llm_response(query, use_db=False, fast_mode=fast_mode)

        best_match = ensemble_results[0]
        context = best_match.page_content if hasattr(best_match, 'page_content') else ""
        metadata = best_match.metadata if hasattr(best_match, 'metadata') else {}

        end_time = time.time()
        processing_time = end_time - start_time
        logging.info(f"처리 시간: {processing_time:.2f}초")

        if fast_mode and processing_time > 20:
            logging.warning("처리 시간이 20초를 초과했습니다. 빠른 모드로 전환합니다.")
            return generate_llm_response(query, context=context, use_db=True, metadata=metadata, fast_mode=True)

        return generate_llm_response(query, context=context, use_db=True, metadata=metadata, fast_mode=fast_mode)
    except Exception as e:
        logging.error(f"쿼리 처리 중 오류 발생: {str(e)}")
        return generate_llm_response(query, use_db=False, fast_mode=fast_mode)

In [ ]:
def check_database():
    try:
        # 컬렉션 정보 가져오기
        collection_info = client.get_collection(COLLECTION_NAME)
        print(f"컬렉션 정보: {collection_info}")

        # 몇 개의 포인트가 있는지 확인
        points_count = client.count(collection_name=COLLECTION_NAME, exact=True)
        print(f"데이터베이스에 저장된 포인트 수: {points_count}")

        # 샘플 포인트 가져오기
        sample_points = client.scroll(
            collection_name=COLLECTION_NAME,
            limit=5,
            with_payload=True,
            with_vectors=False
        )
        print("샘플 포인트:")
        for point in sample_points[0]:
            print(f"ID: {point.id}, Payload: {point.payload}")

    except Exception as e:
        print(f"데이터베이스 확인 중 오류 발생: {str(e)}")

# 스크립트 시작 시 데이터베이스 확인
check_database()

컬렉션 정보: status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> vectors_count=None indexed_vectors_count=0 points_count=2928 segments_count=2 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=768, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=20000, flush_interval_sec=5, max_optimization_threads=None), wal_config=WalConfig(wal_capacity_mb=32, wal_segments_ahead=0), quantization_co

In [ ]:
def langchain_based_rag_search():
    while True:
        query = input("질문을 입력하세요 (빠른 모드는 '빠르게:', 상세 모드는 '상세하게:' 접두사 사용, 종료는 '끝내기'): ").strip()

        if query.lower() == '끝내기':
            print("검색을 종료합니다.")
            break

        fast_mode = True
        if query.startswith("상세하게:"):
            fast_mode = False
            query = query[5:].strip()
        elif query.startswith("빠르게:"):
            query = query[4:].strip()

        if not query:
            print("질문을 입력해주세요.")
            continue

        try:
            answer = process_query(query, fast_mode=fast_mode)
            print(answer)
        except Exception as e:
            logging.error(f"오류 발생: {str(e)}", exc_info=True)
            print(f"오류가 발생했습니다: {str(e)}")
            print("다시 시도해주세요.")

        print("-" * 50)

if __name__ == "__main__":
    langchain_based_rag_search()

질문을 입력하세요 (빠른 모드는 '빠르게:', 상세 모드는 '상세하게:' 접두사 사용, 종료는 '끝내기'): 빠르게: VDT증후군에 대해서 알려줘.


ERROR:root:쿼리 처리 중 오류 발생: unhashable type: 'Document'
ERROR:root:LLM 응답 생성 중 오류 발생: Missing some input keys: {'query'}


죄송합니다. 답변을 생성하는 중에 오류가 발생했습니다: Missing some input keys: {'query'}
--------------------------------------------------
질문을 입력하세요 (빠른 모드는 '빠르게:', 상세 모드는 '상세하게:' 접두사 사용, 종료는 '끝내기'): 끝내기
검색을 종료합니다.
